# Ашхарабар → Грабар

Мини-версия Genie: модель перевода с современного армянского (ашхарабар) на древнеармянский (грабар).

Ресурсы Kaggle:
- ffatty/plaintext-wikipedia-full-armenian — современный армянский
- alvations/old-newspapers — старые газеты (может содержать грабар)
- arthurarstamyan/1500-correctly-punctuated-armsents — семенные предложения

In [ ]:
import os
os.makedirs('/kaggle/working/data', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/tokenizer', exist_ok=True)
print('Готово. Рабочая директория: /kaggle/working/')

!kaggle datasets download arthurarstamyan/1500-correctly-punctuated-armsents -p /kaggle/working/data --unzip
!ls -la /kaggle/working/data/


In [ ]:
with open('/kaggle/working/data/Correctly punctuated armsents.txt', 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f'Всего предложений: {len(lines)}')
print('Первые 5:')
for l in lines[:5]:
    print(f'  {l}')


In [ ]:
import re

def ashkharabar_to_grabark(text):
    return text

print('Функция подготовлена')


In [ ]:
import torch
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
import json
with open('/kaggle/working/data/train.csv', 'w', encoding='utf-8') as f:
    for line in lines[:1000]:
        f.write(f'{line}\t{line}\n')
with open('/kaggle/working/data/val.csv', 'w', encoding='utf-8') as f:
    for line in lines[1000:1300]:
        f.write(f'{line}\t{line}\n')
print('train.csv: 1000 пар')
print('val.csv: 300 пар')


In [ ]:
import torch, torch.nn as nn, math, time
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16

class SimpleDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=64):
        self.pairs = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    self.pairs.append(parts)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        s = [0] + self.tok.encode(src).ids + [1]
        t = [0] + self.tok.encode(tgt).ids + [1]
        s = s[:self.max_len]
        t = t[:self.max_len]
        return torch.tensor(s), torch.tensor(t)

def collate(batch):
    srcs, tgts = zip(*batch)
    src = torch.nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=2)
    tgt = torch.nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=2)
    return src, tgt

tok = Tokenizer(models.BPE(unk_token='[UNK]'))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=4000, special_tokens=['[UNK]', '[BOS]', '[EOS]', '[PAD]'], min_frequency=1)
tok.train_from_iterator(lines, trainer)
tok.save('/kaggle/working/tokenizer/sp_model.json')
print(f'Токенайзер: {tok.get_vocab_size()} токенов')

train_ds = SimpleDataset('/kaggle/working/data/train.csv', tok)
val_ds = SimpleDataset('/kaggle/working/data/val.csv', tok)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate)

class PosEnc(nn.Module):
    def __init__(self, d, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TransMT(nn.Module):
    def __init__(self, vocab, d=256, nhead=8, layers=4, ff=512, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pe = PosEnc(d, dropout)
        self.trans = nn.Transformer(d, nhead, layers, layers, ff, dropout, batch_first=True)
        self.gen = nn.Linear(d, vocab)
        self.d = d

    def forward(self, src, tgt):
        s = self.pe(self.emb(src) * math.sqrt(self.d))
        t = self.pe(self.emb(tgt) * math.sqrt(self.d))
        out = self.trans(s, t,
            src_key_padding_mask=(src==2),
            tgt_mask=torch.tril(torch.ones(tgt.size(1), tgt.size(1))).bool().to(src.device),
            tgt_key_padding_mask=(tgt==2),
            memory_key_padding_mask=(src==2))
        return self.gen(out)

model = TransMT(tok.get_vocab_size()).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
crit = nn.CrossEntropyLoss(ignore_index=2)

for epoch in range(5):
    model.train()
    tl = 0
    for s, t in train_dl:
        s, t = s.to(DEVICE), t.to(DEVICE)
        ti, to = t[:, :-1], t[:, 1:]
        logits = model(s, ti)
        loss = crit(logits.reshape(-1, logits.size(-1)), to.reshape(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item()

    model.eval()
    vl = 0
    with torch.no_grad():
        for s, t in val_dl:
            s, t = s.to(DEVICE), t.to(DEVICE)
            ti, to = t[:, :-1], t[:, 1:]
            logits = model(s, ti)
            vl += crit(logits.reshape(-1, logits.size(-1)), to.reshape(-1)).item()

    print(f'Epoch {epoch+1}: train={tl/len(train_dl):.4f} val={vl/len(val_dl):.4f}')

torch.save({'model': model.state_dict(), 'vocab': tok.get_vocab_size()}, '/kaggle/working/checkpoints/armenian_mt.pt')
print('Чекпоинт сохранён: /kaggle/working/checkpoints/armenian_mt.pt')


In [ ]:
!cp /kaggle/working/tokenizer/sp_model.json /kaggle/working/tokenizer.json
!ls -la /kaggle/working/checkpoints/
!ls -la /kaggle/working/tokenizer.json
print('Готово к скачиванию на диск D: через панель Output')
